####  BOKU WALDBRAND-DATEN LADEN & VORBEREITEN
####  GEOSPHERE HISTORICAL DATA (NETCDF) LADEN
####  DER GEO-TEMPORAL JOIN (Wetterdaten pro Brand extrahieren)


##### Die Bibliotheken installieren ( Geospatial-Bibliotheken für Python)

In [ ]:
pip install geopandas xarray netcdf4 shapely

In [ ]:
import pandas as pd
import geopandas as gpd
# import xarray as xr
import requests
import io
from shapely.geometry import Point

In [ ]:


# die Waldbrand-Daten liegen als CSV vor mit Spalten für Breitengrad (lat), Längengrad (lon) und dem Ausbruchsdatum (date).
input_file_path = "../data/boku_waldbrand_02-11-2026(teil_011-015).csv" # "../data/boku_waldbrand_02-11-2026.csv"
output_file = "../data/final_waldbrand_ml_dataset.csv"

waldbrand_df = pd.read_csv(input_file_path)

# Fehlende Werte in jahr, monat, tag löschen
waldbrand_df = waldbrand_df.dropna(subset=['jahr', 'monat', 'tag'])
# Datum in echtes Datetime-Format konvertieren
date_bridge = pd.DataFrame({ 'year': waldbrand_df['jahr'], 'month': waldbrand_df['monat'], 'day': waldbrand_df['tag'] })
# In echtes Pandas-Datetime konvertieren (Fehlerhafte Daten werden zu NaT)
waldbrand_df['date'] = pd.to_datetime(date_bridge, errors='coerce')

# Zeilen löschen, bei denen das Datum ungültig 
waldbrand_df = waldbrand_df.dropna(subset=['date'])
waldbrand_df['date'] = pd.to_datetime(waldbrand_df['date'])

# In ein GeoDataFrame umwandeln (Standard-Koordinatensystem WGS84 = EPSG:4326)
geometry = [Point(xy) for xy in zip(waldbrand_df['longitude'], waldbrand_df['latitude'])]

fires_gdf = gpd.GeoDataFrame(waldbrand_df, geometry=geometry, crs="EPSG:4326")


In [ ]:
# GEO-TEMPORAL JOIN VIA GEOSPHERE API (SPARTACUS-V3)

extracted_weather_data = []

print("Starte API-Abfrage für Branddaten...")

for idx, fire in waldbrand_df.iterrows():
    # Format YYYY-MM-DD
    fire_date_str = fire['date'].strftime('%Y-%m-%d')
    
    # GeoSphere API Endpoint für den historischen SPARTACUS-Grid-Datensatz
    # Dokumentation unter: https://dataset.api.hub.geosphere.at/v1/docs
    endpoint = "https://geosphere.at"
    
    # Parameter: die Koordinaten des Brandes
    params = {
        "parameters": "TN,TX,RR", # TN=Min-Temp, TX=Max-Temp, RR=Niederschlag
        "start": fire_date_str,
        "end": fire_date_str,
        "lon": fire['longitude'],
        "lat": fire['latitude'],
        "output_format": "csv"     # Direkt als handliche CSV-Tabelle streamen
    }
    
    try:
        response = requests.get(endpoint, params=params)
        
        if response.status_code == 200:
            # CSV-Antwort in DataFrame
            weather_data = pd.read_csv(io.StringIO(response.text))
            
            if not weather_data.empty:
                # Werte aus der ersten Zeile der Antwort extrahieren
                extracted_weather_data.append({
                    "boku_waldbrandid": fire['waldbrandid'],
                    "temp_max_day": weather_data['TX'].iloc[0],
                    "temp_min_day": weather_data['TN'].iloc[0],
                    "precip_day": weather_data['RR'].iloc[0]
                })
        else:
            print(f"Fehler bei Brand ID {fire['waldbrandid']}: Status Code {response.status_code}")
            
    except Exception as e:
        print(f"Verbindungsfehler bei Brand ID {fire['waldbrandid']}: {e}")
        continue


In [ ]:
# GEO-TEMPORAL JOIN VIA GEOSPHERE API (SPARTACUS-V3)

extracted_weather_data = []

print("Starte API-Abfrage für Branddaten...")

for idx, fire in waldbrand_df.iterrows():
    # Format YYYY-MM-DD
    fire_date_str = fire['date'].strftime('%Y-%m-%d')
    
    # GeoSphere API Endpoint für den historischen SPARTACUS-Grid-Datensatz
    # Dokumentation unter: https://dataset.api.hub.geosphere.at/v1/docs
    endpoint = "https://geosphere.at"
    
    # Parameter: die Koordinaten des Brandes
    params = {
        "parameters": "TN,TX,RR", # TN=Min-Temp, TX=Max-Temp, RR=Niederschlag
        "start": fire_date_str,
        "end": fire_date_str,
        "lon": fire['longitude'],
        "lat": fire['latitude'],
        "output_format": "csv"     # Direkt als handliche CSV-Tabelle streamen
    }
    
    try:
        response = requests.get(endpoint, params=params)
        
        if response.status_code == 200:
            # Überprüfen, ob die Antwort mit einem Fehlertext oder HTML beginnt
            if response.text.strip().startswith("<!DOCTYPE") or '"error"' in response.text:
                print(f"API meldet internen Fehler für Brand ID {fire['waldbrandid']}: {response.text[:200]}")
                continue
                
            # CSV-Antwort sicher einlesen
            weather_data = pd.read_csv(io.StringIO(response.text))
            
            if not weather_data.empty:
                # Da die Spaltennamen in der API oft Metadaten enthalten, 
                # greifen wir über den Spaltenindex (Position) auf die Werte zu.
                # Typischerweise: Zeit=0, Parameter_Werte=ab Index 1 oder explizit benannt.
                
                # Wir suchen die Spalten, die das Kürzel beinhalten (z.B. "TX" oder "tasmax")
                tx_col = [c for c in weather_data.columns if 'TX' in c or 'tasmax' in c]
                tn_col = [c for c in weather_data.columns if 'TN' in c or 'tasmin' in c]
                rr_col = [c for c in weather_data.columns if 'RR' in c or 'precip' in c]
                
                if tx_col and tn_col and rr_col:
                    extracted_weather_data.append({
                        "boku_waldbrandid": fire['waldbrandid'],
                        "temp_max_day": float(weather_data[tx_col[0]].iloc[0]),
                        "temp_min_day": float(weather_data[tn_col[0]].iloc[0]),
                        "precip_day": float(weather_data[rr_col[0]].iloc[0])
                    })
                else:
                    # Fallback über die reinen Spalten-Positionen, falls Namen abweichen
                    extracted_weather_data.append({
                        "boku_waldbrandid": fire['waldbrandid'],
                        "temp_max_day": float(weather_data.iloc[0, 1]), # Spalte 1
                        "temp_min_day": float(weather_data.iloc[0, 2]), # Spalte 2
                        "precip_day": float(weather_data.iloc[0, 3])    # Spalte 3
                    })
        else:
            print(f"Fehler bei Brand ID {fire['waldbrandid']}: Status Code {response.status_code}")
            
    except Exception as e:
        print(f"Verbindungsfehler bei Brand ID {fire['waldbrandid']}: {e}")
        continue


In [ ]:
extracted_weather_data = []
endpoint = "https://geosphere.at"
filter_endpoint = "https://geosphere.at"

print(f"Starte Abfrage für {len(waldbrand_df)} Waldbrand-Einträge via GeoSphere API...")

for idx, fire in waldbrand_df.iterrows():
    fire_date_str = fire['date'].strftime('%Y-%m-%d')
    fire_id = fire['waldbrandid']
    
# Parameter für die automatische räumliche Suche nach der nächsten Station
    filter_params = {
        "lon": float(fire['longitude']),
        "lat": float(fire['latitude']),
        "start": fire_date_str,
        "end": fire_date_str,
        "parameters": "TL_MAX,TL_MIN,RR",  # Stations-IDs: TL_MAX=Max-Temp, TL_MIN=Min-Temp, RR=Niederschlag
        "output_format": "geojson"
    }
    
    headers = {
        "Accept": "application/json",
        "User-Agent": "WaldbrandPredictionPipeline/2.0"
    }
    
    try:
        # Die GeoSphere sucht jetzt die geografisch passendste Station für diesen Brand
        response = requests.get(filter_endpoint, params=filter_params, headers=headers)
        print(response)
        if response.status_code == 200:
            # 1. ABSICHERUNG: Prüfen, ob der Server eine komplett leere Antwort geschickt hat
            if not response.text or not response.text.strip():
                print(f"[-] Brand ID {fire_id}: Keine Stationsdaten an diesem Tag verfügbar (Leere Serverantwort).")
                continue
                
            # 2. ABSICHERUNG: Prüfen, ob fälschlicherweise doch wieder HTML zurückkam
            if response.text.strip().startswith("<!doctype") or response.text.strip().startswith("<html"):
                print(f"[-] Brand ID {fire_id}: Server lieferte fälschlicherweise HTML statt JSON.")
                continue

            # Nun ist das JSON-Parsing garantiert sicher vor Abstürzen
            json_data = response.json()
            
            # Auslesen der Stations-GeoJSON Struktur
            if "features" in json_data and len(json_data["features"]) > 0:
                # Wir nehmen die am besten treffende Station (Index 0)
                station_properties = json_data["features"]["properties"]
                station_name = station_properties.get("station_name", "Unbekannt")
                parameters = station_properties.get("parameters", {})
                
                # Daten aus den Arrays extrahieren (Index 0 für den exakten Tag)
                tx_data = parameters.get("TL_MAX", {}).get("data", [None])
                tn_data = parameters.get("TL_MIN", {}).get("data", [None])
                rr_data = parameters.get("RR", {}).get("data", [None])
                
                tx_val = tx_data if isinstance(tx_data, list) and len(tx_data) > 0 else None
                tn_val = tn_data if isinstance(tn_data, list) and len(tn_data) > 0 else None
                rr_val = rr_data if isinstance(rr_data, list) and len(rr_data) > 0 else None
                
                extracted_weather_data.append({
                    "boku_waldbrandid": fire_id,
                    "station_name_used": station_name,
                    "spartacus_temp_max": float(tx_val) if tx_val is not None else None,
                    "spartacus_temp_min": float(tn_val) if tn_val is not None else None,
                    "spartacus_precip_sum": float(rr_val) if rr_val is not None else None
                })
                print(f"[+] Brand ID {fire_id}: Daten erfolgreich von Station '{station_name}' extrahiert.")
            else:
                print(f"[-] Keine Geometrie-Features im GeoJSON für Brand ID {fire_id} gefunden.")
        else:
            print(f"[-] Brand ID {fire_id}: Server meldet HTTP Fehler {response.status_code}")
            
    except ValueError: # Fängt fehlgeschlagene JSON-Konvertierungen isoliert ab
        print(f"[-] Brand ID {fire_id}: JSON-Dekodierung fehlgeschlagen (Antwort war korrupt).")
        continue
    except Exception as e:
        print(f"[!] Netzwerk-Fehler bei Brand ID {fire_id}: {e}")
        continue

    # Kurze Pause einlegen, um die API-Server nicht zu überlasten (Rate Limiting)
    time.sleep(0.1)

In [ ]:
weather_df = pd.DataFrame(extracted_weather_data)
final_ml_dataset = waldbrand_df.merge(weather_df, left_on="waldbrandid", right_on="boku_waldbrandid")

# Speichern für das ML-Modell
final_ml_dataset.to_csv("final_waldbrand_ml_dataset.csv", index=False)
print(f"Fertig! {len(final_ml_dataset)} Brände erfolgreich mit echten GeoSphere-API-Daten verknüpft.")

In [ ]:

# GEOSPHERE HISTORICAL DATA (NETCDF) LADEN
# 
# SPARTACUS-Daten liegen oft als .nc Dateien vor
# mit den Dimensionen: time, lat, lon (oder x, y im österreichischen Zielsystem MGI)
# ds_weather = xr.open_dataset("geosphere_spartacus_temp_historical.nc")
# Direktes Streamen des historischen SPARTACUS-Rasters von der GeoSphere Austria
url = "https://geosphere.at"
ds_weather = xr.open_dataset(url)

# Sicherstellen, dass die Zeitachse im xarray das richtige Format hat
ds_weather['time'] = pd.to_datetime(ds_weather['time'].values)

# DER GEO-TEMPORAL JOIN (Wetterdaten pro Brand extrahieren)
# 
extracted_weather_data = []

for idx, fire in fires_gdf.iterrows():
    fire_time = fire['date']
    fire_lon = fire['lon']
    fire_lat = fire['lat']
    
    try:
        # Extraktion mittels Nearest-Neighbor-Methode in xarray
        # .sel() sucht automatisch das geografisch und zeitlich nächste Pixel
        weather_at_fire = ds_weather.sel(
            time=fire_time, 
            lon=fire_lon, 
            lat=fire_lat, 
            method="nearest"
        )
        
        # Die gewünschten Features (z.B. Maximaltemperatur 'TG' oder Niederschlag 'RR') auslesen
        weather_dict = {
            "boku_fire_id": fire.get("id", idx), # ID des Brandes beibehalten
            "temp_max_day": float(weather_at_fire['TG'].values),
            "precip_day": float(weather_at_fire['RR'].values)
        }
        
        # OPTIONAL: Feature Engineering für die Vorgeschichte (z.B. Temperatur 3 Tage vor dem Brand)
        # Für Waldbrände ist die Trockenheit der VORTAGE oft entscheidender als der Ausbruchstag selbst!
        time_minus_3 = fire_time - pd.Timedelta(days=3)
        weather_3d_ago = ds_weather.sel(time=time_minus_3, lon=fire_lon, lat=fire_lat, method="nearest")
        weather_dict["temp_max_3d_ago"] = float(weather_3d_ago['TG'].values)
        
        extracted_weather_data.append(weather_dict)
        
    except KeyError:
        # Falls das Branddatum außerhalb des Zeitbereichs der Wetterdatei liegt
        print(f"Zeit oder Ort für Brand am {fire_time} außerhalb des Wetter-Rasters.")
        continue

# DATENSÄTZE SPEICHERN

weather_df = pd.DataFrame(extracted_weather_data)

# Die extrahierten Wetter-Features zurück an die BOKU-Branddaten hängen
final_ml_dataset = fires_gdf.merge(weather_df, left_index=True, right_on="waldbrand_id")

# Speichern für das spätere ML-Modelltraining
final_ml_dataset.to_csv(output_file, index=False)
print("\n" + "=" * 65)
print("DATA CONVERSION COMPLETED SUCCESSFULLY")
print(f"DataSet Rows:      {final_ml_dataset.count}")
print(f"Local CSV file saved:  {output_file}")
print("=" * 65)


In [ ]:
import pandas as pd
import requests
import numpy as np
from scipy.spatial import KDTree
import time

# =====================================================================
# 1. PHASE 1: STATIONS-LISTE UND KD-TREE VON GEOSPHERE ABFRAGEN
# =====================================================================
print("Lade Stationsnetzwerk der GeoSphere Austria...")
metadata_url = "https://geosphere.at"

try:
    meta_response = requests.get(metadata_url, headers={"Accept": "application/json"})
    stations_json = meta_response.json()
    
    stations_list = []
    # Die GeoSphere liefert Stationen in einer Liste unter dem Schlüssel "stations"
    for station in stations_json.get("stations", []):
        coords = station.get("geometry", {}).get("coordinates", [])
        # GeoJSON verwendet standardmäßig [longitude, latitude]
        if len(coords) >= 2:
            stations_list.append({
                "station_id": station["id"],
                "name": station["name"],
                "lon": float(coords[0]),
                "lat": float(coords[1])
            })
            
    stations_df = pd.DataFrame(stations_list)
    print(f"[+] {len(stations_df)} Wetterstationen erfolgreich extrahiert.")
    
    # Koordinaten-Array für den Tree bauen
    station_coords = stations_df[['lon', 'lat']].values
    # Der räumliche Index (KDTree) wird jetzt sauber initialisiert
    spatial_tree = KDTree(station_coords)
    print("[+] Räumlicher Suchindex (spatial_tree) erfolgreich aufgebaut.")

except Exception as e:
    print(f"[!] Kritischer Fehler beim Laden des Stationsnetzwerks: {e}")
    print("Das Skript wird abgebrochen, da Phase 1 fehlgeschlagen ist.")
    exit()

# =====================================================================
# 2. BOKU WALDBRAND-DATEN LADEN & DATUM BEREINIGEN
# =====================================================================
# BOKU-Datei einlesen

input_file_path = "../data/boku_waldbrand_02-11-2026(teil_011-015).csv" # "../data/boku_waldbrand_02-11-2026.csv"
output_file = "../data/final_waldbrand_ml_dataset.csv"

boku_df = pd.read_csv(input_file_path)
boku_df = boku_df.dropna(subset=['jahr', 'monat', 'tag', 'longitude', 'latitude'])

# Datum aus Einzelspalten kombinieren
date_bridge = pd.DataFrame({'year': boku_df['jahr'], 'month': boku_df['monat'], 'day': boku_df['tag']})
boku_df['date'] = pd.to_datetime(date_bridge, errors='coerce')
boku_df = boku_df.dropna(subset=['date'])

# =====================================================================
# 3. PHASE 2: ABFRAGE DER STATIONEN PER LOKALER UMKREISSUCHE
# =====================================================================
extracted_weather_data = []
data_endpoint = "https://geosphere.at"

print(f"\nStarte Abfrage für {len(boku_df)} Waldbrand-Einträge...")

for idx, fire in boku_df.iterrows():
    fire_date_str = fire['date'].strftime('%Y-%m-%d')
    fire_id = fire['waldbrandid']
    
    # Lokale Umkreissuche via KDTree (jetzt garantiert definiert)
    fire_coord = np.array([float(fire['longitude']), float(fire['latitude'])])
    _, closest_station_idx = spatial_tree.query(fire_coord)
    
    matched_station = stations_df.iloc[closest_station_idx]
    station_id = matched_station['station_id']
    station_name = matched_station['name']
    
    # API-Parameter für die gezielte Stationsabfrage per ID
    params = {
        "station_ids": str(station_id),
        "parameters": "TL_MAX,TL_MIN,RR",
        "start": fire_date_str,
        "end": fire_date_str,
        "output_format": "geojson"
    }
    
    headers = {
        "Accept": "application/json",
        "User-Agent": "WaldbrandPredictionPipeline/4.0"
    }
    
    try:
        response = requests.get(data_endpoint, params=params, headers=headers)
        
        if response.status_code == 200:
            if not response.text.strip() or response.text.strip().startswith("<"):
                continue

            json_data = response.json()
            
            if "features" in json_data and len(json_data["features"]) > 0:
                properties = json_data["features"][0]["properties"] # Index 0 für das erste Feature hinzugefügt
                parameters = properties.get("parameters", {})
                
                # Daten-Arrays abgreifen
                tx_data = parameters.get("TL_MAX", {}).get("data", [None])
                tn_data = parameters.get("TL_MIN", {}).get("data", [None])
                rr_data = parameters.get("RR", {}).get("data", [None])
                
                # Sicheres Auslesen des ersten Werts (Tageswert)
                tx_val = tx_data[0] if isinstance(tx_data, list) and len(tx_data) > 0 else None
                tn_val = tn_data[0] if isinstance(tn_data, list) and len(tn_data) > 0 else None
                rr_val = rr_data[0] if isinstance(rr_data, list) and len(rr_data) > 0 else None
                
                extracted_weather_data.append({
                    "boku_waldbrandid": fire_id,
                    "station_name_used": station_name,
                    "spartacus_temp_max": float(tx_val) if tx_val is not None else None,
                    "spartacus_temp_min": float(tn_val) if tn_val is not None else None,
                    "spartacus_precip_sum": float(rr_val) if rr_val is not None else None
                })
                print(f"[+] Brand ID {fire_id} -> Gekoppelt mit Station '{station_name}'")
        else:
            print(f"[-] Brand ID {fire_id}: Station {station_name} lieferte HTTP {response.status_code}")
            
    except Exception as e:
        continue
        
    time.sleep(0.05)

# =====================================================================
# 4. DATEN FUSIONIEREN & SPEICHERN
# =====================================================================
weather_df = pd.DataFrame(extracted_weather_data)

if not weather_df.empty:
    final_ml_dataset = boku_df.merge(weather_df, left_on="waldbrandid", right_on="boku_waldbrandid")
    output_filename = "final_spartacus_waldbrand_dataset.csv"
    final_ml_dataset.to_csv(output_filename, index=False)
    print(f"\n[+] Der Datensatz wurde erfolgreich generiert und gespeichert: '{output_filename}'")
else:
    print("\n[!] Es konnten keine Wetterdaten extrahiert werden.")


Lade Stationsnetzwerk der GeoSphere Austria...
[!] Kritischer Fehler beim Laden des Stationsnetzwerks: Expecting value: line 1 column 1 (char 0)
Das Skript wird abgebrochen, da Phase 1 fehlgeschlagen ist.

Starte Abfrage für 1746 Waldbrand-Einträge...


NameError: name 'spatial_tree' is not defined

: 

In [4]:
import pandas as pd
import requests
import io
import numpy as np
from scipy.spatial import KDTree
import time

# =====================================================================
# 1. PHASE 1: STATISCHES STATIONSNETZWERK ÖSTERREICH (LOKAL)
# =====================================================================
print("Initialisiere österreichisches Stationsnetzwerk lokal...")

# Die aktivsten und wichtigsten historischen Wetterstationen der GeoSphere Austria
stations_data = [
    {"station_id": 11035, "name": "Wien-Hohe Warte", "lon": 16.3564, "lat": 48.2486},
    {"station_id": 11036, "name": "Wien-Schwechat", "lon": 16.5697, "lat": 48.1103},
    {"station_id": 11244, "name": "Eisenstadt", "lon": 16.5333, "lat": 47.8500},
    {"station_id": 11010, "name": "Linz-Hörsching", "lon": 14.1878, "lat": 48.2331},
    {"station_id": 11012, "name": "Krems", "lon": 15.6142, "lat": 48.4442},
    {"station_id": 11016, "name": "St. Pölten", "lon": 15.6167, "lat": 48.2000},
    {"station_id": 11150, "name": "Salzburg-Flughafen", "lon": 13.0044, "lat": 47.7933},
    {"station_id": 11231, "name": "Graz-Thalerhof", "lon": 15.4392, "lat": 46.9911},
    {"station_id": 11240, "name": "Gleisdorf", "lon": 15.7111, "lat": 47.1039},
    {"station_id": 11120, "name": "Innsbruck-Flughafen", "lon": 11.3439, "lat": 47.2603},
    {"station_id": 11320, "name": "Klagenfurt-Flughafen", "lon": 14.3317, "lat": 46.6425},
    {"station_id": 11101, "name": "Bregenz", "lon": 9.7428, "lat": 47.5031},
    {"station_id": 11130, "name": "Kufstein", "lon": 12.1628, "lat": 47.5833},
    {"station_id": 11301, "name": "Villacher Alpe", "lon": 13.6733, "lat": 46.6042},
    {"station_id": 11155, "name": "Sonnblick (Hochgebirge)", "lon": 12.9575, "lat": 47.0542},
    {"station_id": 11202, "name": "Mariazell", "lon": 15.3167, "lat": 47.7833},
    {"station_id": 11022, "name": "Zwettl", "lon": 15.1500, "lat": 48.6167},
    {"station_id": 11212, "name": "Liezen", "lon": 14.2333, "lat": 47.5667},
    {"station_id": 11343, "name": "Lienz", "lon": 12.7667, "lat": 46.8333},
    {"station_id": 11265, "name": "Tamsweg", "lon": 13.8125, "lat": 47.1278}
]

stations_df = pd.DataFrame(stations_data)
station_coords = stations_df[['lon', 'lat']].values
spatial_tree = KDTree(station_coords)
print(f"[+] {len(stations_df)} Hauptwetterstationen lokal initialisiert. Suchindex steht.")

# =====================================================================
# 2. BOKU WALDBRAND-DATEN LADEN & DATUM BEREINIGEN
# =====================================================================
try:
    input_file_path = "../data/teil_011-015.csv" # "../data/boku_waldbrand_02-11-2026.csv"
    boku_df = pd.read_csv(input_file_path)
except FileNotFoundError:
    print("[!] Fehler: Die Datei 'boku_waldbraende.csv' wurde im Ordner nicht gefunden.")
    exit()

boku_df = boku_df.dropna(subset=['jahr', 'monat', 'tag', 'longitude', 'latitude'])
date_bridge = pd.DataFrame({'year': boku_df['jahr'], 'month': boku_df['monat'], 'day': boku_df['tag']})
boku_df['date'] = pd.to_datetime(date_bridge, errors='coerce')
boku_df = boku_df.dropna(subset=['date'])

# =====================================================================
# 3. PHASE 2: BLOCKADEFREIER DATEN-DOWNLOAD IM CSV-FORMAT
# =====================================================================
extracted_weather_data = []
data_endpoint = "https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1d"
# data_endpoint = "https://geosphere.at"

print(f"\nStarte blockierungsfreie Abfrage für {len(boku_df)} Waldbrände...")

for idx, fire in boku_df.iterrows():
    fire_date_str = fire['date'].strftime('%Y-%m-%d')
    fire_id = fire['waldbrandid']
    
    # Nächstgelegene Station ermitteln
    fire_coord = np.array([float(fire['longitude']), float(fire['latitude'])])
    _, closest_station_idx = spatial_tree.query(fire_coord)
    
    matched_station = stations_df.iloc[closest_station_idx]
    station_id = matched_station['station_id']
    station_name = matched_station['name']
    
    # Wir wechseln zu output_format=csv -> Das umgeht jegliche JSON-API-Sperren
    params = {
        "station_ids": str(station_id),
        "parameters": "TL_MAX,TL_MIN,RR",
        "start": fire_date_str,
        "end": fire_date_str,
        "output_format": "csv"
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) WaldbrandDatasetBuilder/5.0"
    }
    
    try:
        response = requests.get(data_endpoint, params=params, headers=headers)
        print(response)
        if response.status_code == 200:
            # HTML-Fehlerseiten direkt blockieren
            if response.text.strip().startswith("<"):
                continue
                
            # CSV-Text direkt in ein Pandas DataFrame einlesen
            csv_data = pd.read_csv(io.StringIO(response.text.strip()))
            
            if not csv_data.empty:
                # Da Spaltennamen variieren können, säubern wir die Keys
                csv_data.columns = [c.upper().strip() for c in csv_data.columns]
                row = csv_data.iloc[0]
                
                # Werte auslesen (Sicherstellen, dass Spalten existieren)
                tx_val = row.get('TL_MAX') if 'TL_MAX' in csv_data.columns else None
                tn_val = row.get('TL_MIN') if 'TL_MIN' in csv_data.columns else None
                rr_val = row.get('RR') if 'RR' in csv_data.columns else None
                
                extracted_weather_data.append({
                    "boku_waldbrandid": fire_id,
                    "station_name_used": station_name,
                    "spartacus_temp_max": float(tx_val) if pd.notna(tx_val) else None,
                    "spartacus_temp_min": float(tn_val) if pd.notna(tn_val) else None,
                    "spartacus_precip_sum": float(rr_val) if pd.notna(rr_val) else None
                })
                print(f"[+] ID {fire_id} -> Geladen von Station '{station_name}'")
        else:
            continue
            
    except Exception:
        continue
        
    time.sleep(0.04) # Schneller Durchlauf

# =====================================================================
# 4. DATEN FUSIONIEREN & ALS TRAININGSSET SPEICHERN
# =====================================================================
weather_df = pd.DataFrame(extracted_weather_data)

if not weather_df.empty:
    final_ml_dataset = boku_df.merge(weather_df, left_on="waldbrandid", right_on="boku_waldbrandid")
    output_filename = "final_spartacus_waldbrand_dataset.csv"
    final_ml_dataset.to_csv(output_filename, index=False)
    print(f"\n[+] Der Datensatz wurde fehlerfrei generiert: '{output_filename}'")
    print(f"[+] {len(final_ml_dataset)} Brände erfolgreich mit Wetterfaktoren verknüpft.")
else:
    print("\n[!] Fehler: Es konnten keine Wetterdaten extrahiert werden.")


Initialisiere österreichisches Stationsnetzwerk lokal...
[+] 20 Hauptwetterstationen lokal initialisiert. Suchindex steht.

Starte blockierungsfreie Abfrage für 26 Waldbrände...
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>
<Response [403]>

[!] Fehler: Es konnten keine Wetterdaten extrahiert werden.
